# RAG Bench — 벤치마크 결과 시각화

`run_all_combos.py` 실행 결과를 로드하여 전략별 레이턴시·RAGAS 품질을 시각화한다.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# 한글 폰트 설정 (macOS)
plt.rcParams["font.family"] = "AppleGothic"
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.dpi"] = 120

DATA_DIR = Path("../") / "_benchdata"
print(f"데이터 디렉토리: {DATA_DIR.resolve()}")

## 1. 데이터 로드

In [ ]:
results_df = pd.read_csv(DATA_DIR / "all_combos_results.csv", encoding="utf-8-sig")
ragas_df = pd.read_csv(DATA_DIR / "all_combos_ragas.csv", encoding="utf-8-sig")

with open(DATA_DIR / "qa_dataset.json", encoding="utf-8") as f:
    qa_dataset = json.load(f)

print(f"검색 결과: {len(results_df)}행, 전략 {results_df['strategy'].nunique()}종")
print(f"RAGAS 점수: {len(ragas_df)}행")
print(f"QA 데이터셋: {qa_dataset['num_qa']}개")
ragas_df

## 2. 짧은 레이블 생성

차트 가독성을 위해 전략명을 축약한다.

In [ ]:
def shorten(name: str) -> str:
    """전략명을 차트용 짧은 레이블로 변환한다."""
    replacements = {
        "[1] 한국어 최적 (KoSimCSE + BM25/OKt)": "DS1-KoSimCSE",
        "[2] 다국어 균형 (E5 + SPLADE)": "DS2-E5+SPLADE",
        "[3] 올인원 통합 (BGE-M3)": "DS3-BGE-M3",
        "[4] 경량/빠른 속도 (MiniLM + BM25)": "DS4-MiniLM",
        "[5] 고성능 API (OpenAI Large + SPLADE)": "DS5-OpenAI",
        "[6] 한국어 API (Upstage Solar + BM25/OKt)": "DS6-Upstage",
        "ColBERT (jina-colbert-v2, brute-force)": "ColBERT",
    }
    if name in replacements:
        return replacements[name]
    if name.startswith("ColBERT Rerank"):
        # "ColBERT Rerank ([1] 한국어 최적 ...)" → "Rerank-DS1"
        for k, v in replacements.items():
            if k in name:
                return f"Rerank-{v.split('-',1)[-1] if '-' in v else v}"
        return "Rerank-?"
    return name[:20]

results_df["short"] = results_df["strategy"].map(shorten)
ragas_df["short"] = ragas_df["strategy"].map(shorten)

# 전략 타입 분류
def classify(name: str) -> str:
    if name.startswith("Rerank"):
        return "ColBERTRerank"
    if name == "ColBERT":
        return "ColBERT"
    return "DenseSparse"

results_df["type"] = results_df["short"].map(classify)
ragas_df["type"] = ragas_df["short"].map(classify)

TYPE_COLORS = {"DenseSparse": "#4C78A8", "ColBERT": "#E45756", "ColBERTRerank": "#72B7B2"}

ragas_df[["short", "type"]].values.tolist()

## 3. 평균 레이턴시 비교

In [ ]:
lat = (
    results_df.groupby(["short", "type"], sort=False)["latency_ms"]
    .mean()
    .reset_index()
    .sort_values("latency_ms")
)

fig, ax = plt.subplots(figsize=(10, 5))
colors = [TYPE_COLORS[t] for t in lat["type"]]
bars = ax.barh(lat["short"], lat["latency_ms"], color=colors)

for bar, val in zip(bars, lat["latency_ms"]):
    ax.text(bar.get_width() + 100, bar.get_y() + bar.get_height() / 2,
            f"{val:,.0f}ms", va="center", fontsize=9)

ax.set_xlabel("평균 레이턴시 (ms)")
ax.set_title("전략별 평균 검색 레이턴시")

# 범례
from matplotlib.patches import Patch
legend_handles = [Patch(color=c, label=l) for l, c in TYPE_COLORS.items()]
ax.legend(handles=legend_handles, loc="lower right")

plt.tight_layout()
plt.show()

## 4. RAGAS 메트릭 비교 (Grouped Bar)

In [ ]:
metrics = ["faithfulness", "answer_relevancy", "context_precision", "context_recall"]
metric_labels = ["Faithfulness", "Answer\nRelevancy", "Context\nPrecision", "Context\nRecall"]

strategies = ragas_df["short"].tolist()
n_strategies = len(strategies)
n_metrics = len(metrics)
x = np.arange(n_metrics)
width = 0.8 / n_strategies

fig, ax = plt.subplots(figsize=(12, 6))

for i, (_, row) in enumerate(ragas_df.iterrows()):
    vals = [row[m] for m in metrics]
    color = TYPE_COLORS[row["type"]]
    offset = (i - n_strategies / 2 + 0.5) * width
    ax.bar(x + offset, vals, width * 0.9, label=row["short"], color=color, alpha=0.6 + 0.4 * (i % 2))

ax.set_xticks(x)
ax.set_xticklabels(metric_labels, fontsize=10)
ax.set_ylabel("점수 (0~1)")
ax.set_ylim(0, 1.15)
ax.set_title("RAGAS 메트릭별 전략 비교")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
ax.axhline(y=1.0, color="gray", linestyle="--", linewidth=0.5, alpha=0.5)

plt.tight_layout()
plt.show()

## 5. RAGAS 레이더 차트

In [ ]:
angles = np.linspace(0, 2 * np.pi, n_metrics, endpoint=False).tolist()
angles += angles[:1]  # 닫기

fig, ax = plt.subplots(figsize=(7, 7), subplot_kw=dict(polar=True))

for _, row in ragas_df.iterrows():
    vals = [row[m] for m in metrics]
    vals += vals[:1]
    color = TYPE_COLORS[row["type"]]
    ax.plot(angles, vals, "o-", linewidth=1.5, label=row["short"], color=color,
            alpha=0.7, markersize=4)
    ax.fill(angles, vals, alpha=0.05, color=color)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(metric_labels, fontsize=9)
ax.set_ylim(0, 1.1)
ax.set_title("RAGAS 레이더 차트", pad=20)
ax.legend(bbox_to_anchor=(1.25, 1.05), loc="upper left", fontsize=7)

plt.tight_layout()
plt.show()

## 6. 품질-속도 트레이드오프 (Scatter)

In [ ]:
# RAGAS 종합 점수 = 4개 메트릭 평균
ragas_df["ragas_avg"] = ragas_df[metrics].mean(axis=1)

lat_avg = results_df.groupby("short", sort=False)["latency_ms"].mean()
merged = ragas_df.set_index("short").join(lat_avg.rename("avg_latency")).reset_index()

fig, ax = plt.subplots(figsize=(10, 6))

for _, row in merged.iterrows():
    color = TYPE_COLORS[row["type"]]
    ax.scatter(row["avg_latency"], row["ragas_avg"], s=150, c=color,
              edgecolors="black", linewidths=0.5, zorder=3)
    ax.annotate(row["short"], (row["avg_latency"], row["ragas_avg"]),
               textcoords="offset points", xytext=(8, 5), fontsize=8)

ax.set_xlabel("평균 레이턴시 (ms)")
ax.set_ylabel("RAGAS 종합 점수 (4개 메트릭 평균)")
ax.set_title("품질 vs 속도 트레이드오프")

# 이상적 영역 표시
ax.annotate("← 빠르고 좋음", xy=(0.02, 0.98), xycoords="axes fraction",
           fontsize=10, color="green", alpha=0.6, weight="bold")
ax.annotate("느리고 나쁨 →", xy=(0.80, 0.02), xycoords="axes fraction",
           fontsize=10, color="red", alpha=0.6, weight="bold")

# 범례
legend_handles = [Patch(color=c, label=l) for l, c in TYPE_COLORS.items()]
ax.legend(handles=legend_handles, loc="lower left")
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. RAGAS 히트맵

In [ ]:
heatmap_data = ragas_df.set_index("short")[metrics]

fig, ax = plt.subplots(figsize=(8, 5))
im = ax.imshow(heatmap_data.values, cmap="RdYlGn", aspect="auto", vmin=0, vmax=1)

ax.set_xticks(range(n_metrics))
ax.set_xticklabels(metric_labels, fontsize=9)
ax.set_yticks(range(len(heatmap_data)))
ax.set_yticklabels(heatmap_data.index, fontsize=9)

# 셀 값 표시
for i in range(len(heatmap_data)):
    for j in range(n_metrics):
        val = heatmap_data.values[i, j]
        color = "white" if val < 0.5 else "black"
        ax.text(j, i, f"{val:.2f}", ha="center", va="center", fontsize=9, color=color)

fig.colorbar(im, ax=ax, label="점수")
ax.set_title("RAGAS 점수 히트맵")

plt.tight_layout()
plt.show()

## 8. 쿼리별 레이턴시 분포

In [ ]:
queries = results_df["query"].unique()
n_queries = len(queries)

fig, axes = plt.subplots(1, n_queries, figsize=(7 * n_queries, 5), sharey=True)
if n_queries == 1:
    axes = [axes]

for idx, query in enumerate(queries):
    ax = axes[idx]
    subset = results_df[results_df["query"] == query].sort_values("latency_ms")
    colors = [TYPE_COLORS[t] for t in subset["type"]]
    ax.barh(subset["short"], subset["latency_ms"], color=colors)
    ax.set_xlabel("레이턴시 (ms)")
    ax.set_title(f"Q{idx+1}: {query[:40]}...", fontsize=10)

plt.suptitle("쿼리별 레이턴시 분포", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 9. 종합 순위표

In [ ]:
ranking = merged[["short", "type", "avg_latency", "ragas_avg"] + metrics].copy()
ranking = ranking.rename(columns={
    "short": "전략",
    "type": "분류",
    "avg_latency": "평균 레이턴시(ms)",
    "ragas_avg": "RAGAS 종합",
})
ranking = ranking.sort_values("RAGAS 종합", ascending=False)
ranking = ranking.reset_index(drop=True)
ranking.index = ranking.index + 1
ranking.index.name = "순위"

# 스타일링
def highlight_best(s):
    if s.dtype == "float64":
        is_best = s == s.max() if s.name != "평균 레이턴시(ms)" else s == s.min()
        return ["font-weight: bold; background-color: #d4edda" if v else "" for v in is_best]
    return ["" for _ in s]

(
    ranking.style
    .apply(highlight_best)
    .format({col: "{:.4f}" for col in metrics + ["RAGAS 종합"]})
    .format({"평균 레이턴시(ms)": "{:,.0f}"})
)